In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import mainStartingFive, teamStarPlayer, projectedStartingFive

### Load Model

In [2]:
q10_model = joblib.load('../MODELS/SAVED_MODELS/xgb_q10_modelv2.pkl')
q50_model = joblib.load('../MODELS/SAVED_MODELS/xgb_q50_modelv2.pkl')
q90_model = joblib.load('../MODELS/SAVED_MODELS/xgb_q90_modelv2.pkl')
models = {'q10': q10_model, 'q50': q50_model, 'q90': q90_model}
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [3]:

pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Top EVs for single bets

In [4]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, models, features, current_date, edge_threshold=0.10, stake=10, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=False, n_simulations=10000, max_kelly=0.25)

singleBets = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
# singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets_{today}.csv', index=False)
singleBets.head(15)

Processing single bets with quantile models...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,Q10,Q90,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,MODEL PROB,EDGE,EV%,KELLY_FRACTION,KELLY_DOLLARS,CONFIDENCE INTERVAL,SIGMA,SIMULATION_METHOD
0,Ben Saraf,BetRivers,player_points,5.5,114,over,8.16,1.51,17.41,1,0.651,0.349,0.467,0.651,0.184,3.94,0.345,2.85,"(1.5, 17.4)",6.83,Analytical
1,Miles McBride,Bovada,player_points,7.5,175,over,7.53,1.25,16.38,1,0.502,0.498,0.364,0.502,0.138,3.81,0.217,3.81,"(1.2, 16.4)",6.50,Analytical
2,Ryan Kalkbrenner,Bovada,player_points,7.5,185,over,7.22,1.49,16.19,1,0.483,0.517,0.351,0.483,0.132,3.75,0.203,3.75,"(1.5, 16.2)",6.32,Analytical
3,Ben Saraf,DraftKings,player_points,4.5,-105,over,8.16,1.51,17.41,1,0.704,0.296,0.512,0.704,0.192,3.74,0.393,2.38,"(1.5, 17.4)",6.83,Analytical
4,Ben Saraf,FanDuel,player_points,4.5,-106,over,8.16,1.51,17.41,1,0.704,0.296,0.515,0.704,0.189,3.68,0.390,2.36,"(1.5, 17.4)",6.83,Analytical
5,Kel'el Ware,Bovada,player_points,8.5,165,over,8.62,1.15,17.81,1,0.507,0.493,0.377,0.507,0.129,3.43,0.208,3.43,"(1.1, 17.8)",7.16,Analytical
6,Harrison Barnes,Bovada,player_points,7.5,175,over,7.30,1.02,15.74,1,0.487,0.513,0.364,0.487,0.123,3.40,0.194,3.40,"(1.0, 15.7)",6.32,Analytical
7,Ryan Rollins,Bovada,player_points,8.5,165,over,8.29,1.49,17.20,1,0.488,0.512,0.377,0.488,0.110,2.92,0.177,2.92,"(1.5, 17.2)",6.75,Analytical
8,Ben Saraf,BetOnline.ag,player_points,5.5,-105,over,8.16,1.51,17.41,1,0.651,0.349,0.512,0.651,0.139,2.72,0.285,2.38,"(1.5, 17.4)",6.83,Analytical
9,Ben Saraf,DraftKings,player_points,4.5,-125,over,8.16,1.51,17.41,1,0.704,0.296,0.556,0.704,0.148,2.67,0.333,2.00,"(1.5, 17.4)",6.83,Analytical


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s25, dfsPTS, models, features, edge_threshold=0.20, stake=100, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
# underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs_{today}.csv', index=False)
underdogPairs.head()

,player1,player2,line1,line2,pred1,pred2,q10_1,q90_1,q10_2,q90_2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,simulation_method
0,Bennedict Mathurin,Isaac Jones,24.5,4.5,15.83,1.41,6.41,25.68,0.00,7.84,under,under,0.858,0.829,0.7116,0.280,0.251,0.266,1.13,0.567,1,Monte Carlo
1,Bennedict Mathurin,Ochai Agbaji,24.5,4.5,15.83,10.81,6.41,25.68,3.74,19.61,under,over,0.858,0.819,0.7030,0.280,0.241,0.261,1.11,0.555,1,Monte Carlo
2,Josh Hart,Bennedict Mathurin,8.5,24.5,14.71,15.83,6.00,22.43,6.41,25.68,over,under,0.807,0.858,0.6927,0.229,0.280,0.255,1.08,0.539,1,Monte Carlo
3,Bennedict Mathurin,Matisse Thybulle,24.5,3.5,15.83,8.60,6.41,25.68,1.90,15.26,under,over,0.858,0.810,0.6950,0.280,0.232,0.256,1.08,0.542,1,Monte Carlo
4,Ochai Agbaji,Isaac Jones,4.5,4.5,10.81,1.41,3.74,19.61,0.00,7.84,over,under,0.819,0.829,0.6796,0.241,0.251,0.246,1.04,0.519,1,Monte Carlo


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s25, dfsPTS, models, features, edge_threshold=0.20, stake=100, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs_{today}.csv', index=False)
pairsPrizepicks.head()

Processing pairs...


,PLAYER 1,PLAYER 2,CATEGORY,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL_SIDE 1,MODEL_SIDE 2,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Aaron Gordon,Anthony Black,player_points,14.5,8.5,17.22,15.87,OVER,OVER,0.629,0.371,0.950,0.050,"(1.3, 33.3)","(7.7, 25.3)",OVER/OVER,0,0.5976,0.793,0.396
1,Aaron Gordon,Mikal Bridges,player_points,14.5,15.5,17.22,4.80,OVER,UNDER,0.629,0.371,0.050,0.950,"(1.3, 33.3)","(0.0, 13.5)",OVER/UNDER,0,0.5976,0.793,0.396
2,Aaron Gordon,Aaron Wiggins,player_points,14.5,13.5,17.22,20.27,OVER,OVER,0.629,0.371,0.950,0.050,"(1.3, 33.3)","(14.9, 25.7)",OVER/OVER,0,0.5976,0.793,0.396
3,Aaron Gordon,Luke Kennard,player_points,14.5,5.5,17.22,11.38,OVER,OVER,0.629,0.371,0.920,0.080,"(1.3, 33.3)","(3.7, 19.5)",OVER/OVER,0,0.5787,0.736,0.368
4,Aaron Gordon,Goga Bitadze,player_points,14.5,4.5,17.22,8.54,OVER,OVER,0.629,0.371,0.809,0.191,"(1.3, 33.3)","(0.0, 17.0)",OVER/OVER,0,0.5089,0.527,0.263


### DraftKings Pick 6 picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'DraftKings Pick6') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s25, dfsPTS, models, features, edge_threshold=0.20, stake=100, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
                     
pairsDraftKings = results.sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsDraftKings.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/draftKingsPairs_{today}.csv', index=False)
pairsDraftKings.head()

Processing pairs...


,PLAYER 1,PLAYER 2,CATEGORY,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL_SIDE 1,MODEL_SIDE 2,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Aaron Gordon,Aaron Wiggins,player_points,14.5,13.5,17.22,20.27,OVER,OVER,0.628,0.372,0.950,0.050,"(1.6, 33.3)","(14.1, 26.4)",OVER/OVER,0,0.5966,0.790,0.395
1,Aaron Gordon,Bruce Brown,player_points,14.5,5.5,17.22,10.74,OVER,OVER,0.628,0.372,0.828,0.172,"(1.6, 33.3)","(0.0, 22.5)",OVER/OVER,0,0.5200,0.560,0.280
2,Aaron Gordon,Jonathan Kuminga,player_points,14.5,16.5,17.22,11.80,OVER,UNDER,0.628,0.372,0.209,0.791,"(1.6, 33.3)","(0.1, 23.5)",OVER/UNDER,0,0.4967,0.490,0.245
3,Aaron Gordon,Bennedict Mathurin,player_points,14.5,18.5,17.22,23.95,OVER,OVER,0.628,0.372,0.764,0.236,"(1.6, 33.3)","(8.4, 40.4)",OVER/OVER,0,0.4798,0.439,0.220
4,Aaron Gordon,Aaron Nesmith,player_points,14.5,14.5,17.22,11.73,OVER,UNDER,0.628,0.372,0.252,0.748,"(1.6, 33.3)","(3.9, 19.9)",OVER/UNDER,0,0.4697,0.409,0.205


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate2LegBets(s25, dfsPTS, models, features, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios_{today}.csv', index=False)
underdogTrios.head()

Processing 3-leg parlays...


,PLAYER 1,PLAYER 2,PLAYER 3,CATEGORY,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_SIDE 1,MODEL_SIDE 2,MODEL_SIDE 3,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Anthony Black,Jock Landale,Kentavious Caldwell-Pope,player_points,8.5,8.5,9.5,15.87,3.28,4.50,OVER,UNDER,UNDER,0.95,0.05,0.095,0.905,0.050,0.950,"(7.1, 24.8)","(0.0, 11.2)","(0.0, 9.9)",OVER/UNDER/UNDER,0,0.8167,3.900,0.780
1,Anthony Black,Jaylen Wells,Kentavious Caldwell-Pope,player_points,8.5,11.5,9.5,15.87,7.11,4.50,OVER,UNDER,UNDER,0.95,0.05,0.161,0.839,0.050,0.950,"(7.1, 24.8)","(0.0, 15.7)","(0.0, 9.9)",OVER/UNDER/UNDER,0,0.7568,3.541,0.708
2,Anthony Black,Bruce Brown,Kentavious Caldwell-Pope,player_points,8.5,5.5,9.5,15.87,10.74,4.50,OVER,OVER,UNDER,0.95,0.05,0.811,0.189,0.050,0.950,"(7.1, 24.8)","(0.0, 22.6)","(0.0, 9.9)",OVER/OVER/UNDER,0,0.7322,3.393,0.679
3,Anthony Black,Kentavious Caldwell-Pope,Tobias Harris,player_points,8.5,9.5,14.5,15.87,4.50,11.01,OVER,UNDER,UNDER,0.95,0.05,0.050,0.950,0.199,0.801,"(7.1, 24.8)","(0.0, 9.9)","(2.8, 19.2)",OVER/UNDER/UNDER,0,0.7227,3.336,0.667
4,Anthony Black,Jaylen Wells,Jock Landale,player_points,8.5,11.5,8.5,15.87,7.11,3.28,OVER,UNDER,UNDER,0.95,0.05,0.161,0.839,0.095,0.905,"(7.1, 24.8)","(0.0, 15.7)","(0.0, 11.2)",OVER/UNDER/UNDER,0,0.7209,3.325,0.665


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate2LegBets(s25, dfsPTS, models, features, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios_{today}.csv', index=False)
triosPrizepicks.head()

Processing 3-leg parlays...


,PLAYER 1,PLAYER 2,PLAYER 3,CATEGORY,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_SIDE 1,MODEL_SIDE 2,MODEL_SIDE 3,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Aaron Wiggins,Anthony Black,Mikal Bridges,player_points,13.5,8.5,15.5,20.27,15.87,4.80,OVER,OVER,UNDER,0.950,0.050,0.948,0.052,0.050,0.950,"(14.2, 26.4)","(6.9, 24.9)","(0.0, 13.8)",OVER/OVER/UNDER,1,0.8556,4.133,0.827
1,Aaron Wiggins,Luke Kennard,Mikal Bridges,player_points,13.5,5.5,15.5,20.27,11.38,4.80,OVER,OVER,UNDER,0.950,0.050,0.926,0.074,0.050,0.950,"(14.2, 26.4)","(3.4, 19.5)","(0.0, 13.8)",OVER/OVER/UNDER,1,0.8354,4.013,0.803
2,Aaron Wiggins,Anthony Black,Luke Kennard,player_points,13.5,8.5,5.5,20.27,15.87,11.38,OVER,OVER,OVER,0.950,0.050,0.948,0.052,0.926,0.074,"(14.2, 26.4)","(6.9, 24.9)","(3.4, 19.5)",OVER/OVER/OVER,1,0.8337,4.002,0.800
3,Anthony Black,Luke Kennard,Mikal Bridges,player_points,8.5,5.5,15.5,15.87,11.38,4.80,OVER,OVER,UNDER,0.948,0.052,0.926,0.074,0.050,0.950,"(6.9, 24.9)","(3.4, 19.5)","(0.0, 13.8)",OVER/OVER/UNDER,1,0.8337,4.002,0.800
4,Aaron Wiggins,Goga Bitadze,Mikal Bridges,player_points,13.5,4.5,15.5,20.27,8.54,4.80,OVER,OVER,UNDER,0.950,0.050,0.814,0.186,0.050,0.950,"(14.2, 26.4)","(0.0, 17.6)","(0.0, 13.8)",OVER/OVER/UNDER,0,0.7349,3.409,0.682


### DraftKings Pick 6 picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'DraftKings Pick6') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate2LegBets(s25, dfsPTS, models, features, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='skew_t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosDraftKings = threeLeg.sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosDraftKings.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/draftKingsTrios_{today}.csv', index=False)
triosDraftKings.head()

Processing 3-leg parlays...


,PLAYER 1,PLAYER 2,PLAYER 3,CATEGORY,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_SIDE 1,MODEL_SIDE 2,MODEL_SIDE 3,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Aaron Wiggins,Bruce Brown,Jonathan Kuminga,player_points,13.5,5.5,16.5,20.27,10.74,11.80,OVER,OVER,UNDER,0.95,0.05,0.807,0.193,0.198,0.802,"(14.4, 26.3)","(0.0, 22.7)","(0.4, 22.8)",OVER/OVER/UNDER,0,0.6152,2.691,0.538
1,Aaron Wiggins,Bennedict Mathurin,Bruce Brown,player_points,13.5,18.5,5.5,20.27,23.95,10.74,OVER,OVER,OVER,0.95,0.05,0.752,0.248,0.807,0.193,"(14.4, 26.3)","(8.2, 39.5)","(0.0, 22.7)",OVER/OVER/OVER,0,0.5769,2.461,0.492
2,Aaron Wiggins,Bruce Brown,Christian Braun,player_points,13.5,5.5,12.5,20.27,10.74,14.77,OVER,OVER,OVER,0.95,0.05,0.807,0.193,0.751,0.249,"(14.4, 26.3)","(0.0, 22.7)","(8.1, 21.4)",OVER/OVER/OVER,0,0.5761,2.457,0.491
3,Aaron Wiggins,Bruce Brown,Shai Gilgeous-Alexander,player_points,13.5,5.5,32.5,20.27,10.74,28.06,OVER,OVER,UNDER,0.95,0.05,0.807,0.193,0.251,0.749,"(14.4, 26.3)","(0.0, 22.7)","(15.0, 41.5)",OVER/OVER/UNDER,0,0.5745,2.447,0.489
4,Aaron Wiggins,Bennedict Mathurin,Jonathan Kuminga,player_points,13.5,18.5,16.5,20.27,23.95,11.80,OVER,OVER,UNDER,0.95,0.05,0.752,0.248,0.198,0.802,"(14.4, 26.3)","(8.2, 39.5)","(0.4, 22.8)",OVER/OVER/UNDER,0,0.5730,2.438,0.488
